# CSC 2301 · Week 3 — Functions: reusable engineering formulas

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/csc2301/notebooks/CSC2301_W3_functions_block_model.ipynb)

**IUGB · BSc Mining / Petroleum Engineering** · Instructor: Laurent Kouadio

Engineers use the same formulas again and again: tonnage of a block, metal content, pressure at depth.
A **function** gives a formula a name, so you write it once, test it once and reuse it everywhere.
The data are a small **block model** of an open-pit gold mine (synthetic): 120 blocks of 10 m × 10 m × 5 m,
each with a density and a gold grade.

**What you will practise:** `def` and `return` · parameters and arguments · default values ·
returning several results · docstrings · calling a function from another function · local variables.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/csc2301/'
FILES = [('data/block_model_pit1.csv', 'block_model_pit1.csv'), ('tests/w3/q1.py', 'tests/q1.py'), ('tests/w3/q2.py', 'tests/q2.py'), ('tests/w3/q3.py', 'tests/q3.py'), ('tests/w3/q4.py', 'tests/q4.py'), ('tests/w3/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

## 1. Why functions?

Without a function, the tonnage formula is copied every time it is needed, and every copy is a chance
to make a mistake. With a function, the formula lives in **one place**.

In [ ]:
def block_volume(length_m, width_m, height_m):
    """Volume of a rectangular block, in m³."""
    return length_m * width_m * height_m

v = block_volume(10, 10, 5)
print(v, "m³")
help(block_volume)      # the docstring is the function's user manual

* `def` starts the definition; the **parameters** are the names in brackets.
* When you **call** the function, the **arguments** (10, 10, 5) are given to the parameters in order.
* `return` sends the result back. A function without `return` gives `None`.
* Variables created inside a function are **local**: they disappear when the function ends.

In [ ]:
def stripping_ratio(waste_t, ore_t):
    ratio = waste_t / ore_t     # 'ratio' only exists inside the function
    return ratio

print(stripping_ratio(4_500_000, 1_500_000))   # 3 t of waste per t of ore
# print(ratio)   # would fail: NameError, ratio is local

## 2. Default values and keyword arguments

A parameter can have a **default value**, used when the caller does not give one.
Arguments can also be passed **by name**, which makes long calls easier to read.

In [ ]:
def mud_weight_ppg(density_g_cm3=1.20):
    """Drilling mud density in pounds per gallon (1 g/cm³ = 8.345 ppg)."""
    return density_g_cm3 * 8.345

print(round(mud_weight_ppg(), 2))                     # uses the default 1.20
print(round(mud_weight_ppg(1.45), 2))                 # positional argument
print(round(mud_weight_ppg(density_g_cm3=1.45), 2))   # keyword argument

## 3. Returning several results

`return a, b` returns a **tuple**. The caller can unpack it into two names.

In [ ]:
def min_max(values):
    return min(values), max(values)

low, high = min_max([0.2, 1.4, 3.1, 0.8])
print(low, high)

## 4. The block model

In [ ]:
import csv

with open("block_model_pit1.csv", encoding="utf-8") as f:
    blocks = list(csv.DictReader(f))

print(len(blocks), "blocks")
print(blocks[0])

---
## Exercises

### Q1. Tonnage of a block

Write `block_tonnes(length_m, width_m, height_m, density_t_m3)` that returns the tonnage of a block.
You may reuse `block_volume` from section 1.

In [ ]:
def block_tonnes(length_m, width_m, height_m, density_t_m3):
    ...

block_tonnes(10, 10, 5, 2.7)

In [ ]:
grader.check("q1")

### Q2. Gold content in ounces

Gold is sold in **troy ounces**: 1 oz = 31.1035 g. A grade of 1 g/t means 1 gram of gold per tonne of rock.
Write `gold_ounces(tonnes, grade_gpt)` that returns the ounces of gold in `tonnes` of rock at `grade_gpt`.

In [ ]:
def gold_ounces(tonnes, grade_gpt):
    ...

gold_ounces(1350, 2.0)

In [ ]:
grader.check("q2")

### Q3. A default value: pressure at depth

The **hydrostatic pressure** of a column of fluid, in MPa, is

$$P = \frac{\text{depth (m)} \times \text{density (g/cm³)} \times 9.81}{1000}$$

Write `hydrostatic_mpa(depth_m, density_g_cm3=1.03)`. The default 1.03 g/cm³ is sea water;
drillers pass the mud density instead.

In [ ]:
def hydrostatic_mpa(depth_m, density_g_cm3=1.03):
    ...

hydrostatic_mpa(1000), hydrostatic_mpa(2000, 1.20)

In [ ]:
grader.check("q3")

### Q4. One function calling two others

Write `block_summary(row)` that takes **one row** of the block model (a dictionary read from the CSV)
and returns a tuple `(tonnes, ounces)`. Reuse `block_tonnes` and `gold_ounces`; remember that values
read from a CSV file are text.

In [ ]:
def block_summary(row):
    ...

block_summary(blocks[0])

In [ ]:
grader.check("q4")

### Q5. Ore above the cut-off

A block is **ore** when its grade is at least the **cut-off grade**, here 0.5 g/t. Below, it is waste.
Using your functions, store in `ore_tonnes` the total tonnes of ore and in `ore_ounces` the total
ounces of gold they contain (both floats).

In [ ]:
CUTOFF = 0.5
ore_tonnes = 0.0
ore_ounces = 0.0
...
ore_tonnes, ore_ounces

In [ ]:
grader.check("q5")

### Going further (not checked)

* Turn the cut-off into a parameter: write `ore_above(blocks, cutoff)` that returns `(tonnes, ounces)`,
  and print the results for cut-offs of 0.3, 0.5, 0.8 and 1.0 g/t. This table is a **grade–tonnage curve**.
* Add a docstring to each of your functions and read it back with `help()`.

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()